# 02 · Train tagger (encoder-only, gán nhãn từng ký tự)

Cần **GPU T4**. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Input: code + output của notebook 01 (Kaggle: add `vi-diacritics-code` và output notebook 01 làm input;
Colab: chạy notebook 01 trước, `data/splits` nằm sẵn trên Drive).
~5M tham số, 6 layer, d_model=256. Chạy ~5 tiếng, tự dừng và lưu checkpoint trước khi hết giờ.
Colab free hay bị ngắt nên mỗi lần chỉ train 150 phút, checkpoint `runs/tagger/last.pt` nằm trên Drive:
mở session mới chạy lại notebook là train tiếp từ checkpoint (`--resume`).

In [ ]:
import glob, json, os, sys

IN_KAGGLE = os.path.exists("/kaggle/input")
IN_COLAB = "google.colab" in sys.modules
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_diacritics/__init__.py", recursive=True)[0].rsplit("/vi_diacritics", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-diacritics-transformer" if IN_COLAB else os.path.abspath("vi-diacritics-transformer")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-diacritics-transformer {REPO}
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vi-diacritics-transformer" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in ("/kaggle/input", WORK, REPO):
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


SPLITS = os.path.dirname(find("splits/train.txt", "chạy notebook 01 trước"))
# Kaggle cho chạy liền 12h; Colab free hay ngắt sau vài tiếng nên mỗi lần chỉ chạy 150 phút,
# last.pt lưu trên Drive, chạy lại notebook là --resume train tiếp
MAX_MIN = 150 if IN_COLAB else 300
print(CODE, SPLITS)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q onnx onnxruntime

In [ ]:
!python -m vi_diacritics.train --task tagger --data {SPLITS} --out runs/tagger \
    --layers 6 --d-model 256 --heads 8 --d-ff 1024 --batch-size 128 --lr 5e-4 \
    --epochs 3 --eval-every 2000 --max-minutes {MAX_MIN} --resume

## Đường loss và accuracy trên val

In [ ]:
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open("runs/tagger/log.jsonl", encoding="utf-8")]
tr = [r for r in rows if "loss" in r]
ev = [r["eval"] for r in rows if "eval" in r]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r["step"] for r in tr], [r["loss"] for r in tr])
ax[0].set_yscale("log"); ax[0].set_title("train loss"); ax[0].set_xlabel("step")
ax[1].plot([e["step"] for e in ev], [e["word_acc"] for e in ev], label="word acc")
ax[1].plot([e["step"] for e in ev], [e["sent_acc"] for e in ev], label="sent acc")
ax[1].legend(); ax[1].set_title("val"); ax[1].set_xlabel("step")
fig.savefig("runs/tagger/curves.png", dpi=110, bbox_inches="tight")

## Xuất ONNX (fp32 + int8)

In [ ]:
!python -m vi_diacritics.export_onnx --run runs/tagger --out export
!ls -la export

## Attention: "con ma" vs "mà thôi" vs "ma túy"
Ký tự 'a' trong "ma" nhìn vào đâu để quyết định dấu?

In [ ]:
import torch
from vi_diacritics.model import TaggerTransformer
from vi_diacritics.viz import plot_attention
from vi_diacritics.vocab import CharVocab

cfg = json.load(open("runs/tagger/config.json"))
vocab = CharVocab.load("runs/tagger/vocab.json")
model = TaggerTransformer(len(vocab), cfg["n_labels"], torch.tensor(vocab.label_mask()), cfg["d_model"],
                          cfg["heads"], cfg["layers"], cfg["d_ff"], 0.0)
model.load_state_dict(torch.load("runs/tagger/best.pt", map_location="cpu")["model"])

for i, s in enumerate(["nhà có con ma", "ma tôi không biết", "ma túy là chất cấm"]):
    pos = s.index("ma") + 1
    plot_attention(model, vocab, s, layers=(0, 2, 5), query_char=pos, path=f"runs/tagger/attn_{i}.png")